# S3: native code-mixed training, and all three systems on English

The third system. **S1** trains on English only. **S2** starts from S1 and trains
LoRA adapters on code-mixed audio. **S3** never sees English spoofing data: it
fine-tunes the pretrained wav2vec2 encoder directly on code-mixed audio (P-028).

Each S3 run reads exactly the manifests of the matching S2 adapter from the W10 run,
so an S2 vs S3 difference is the training method and nothing else:

| Run | Data | Condition | Matching S2 adapter |
|---|---|---|---|
| `s3_native_clean` | XTTS | clean | `lora_norm_clean` |
| `s3_native_channel` | XTTS | G.711 @ 20 dB | `lora_norm_channel` |
| `s3_native_rvc_clean` | XTTS + RVC training half | clean | `lora_norm_rvc_clean` |
| `s3_native_rvc_channel` | XTTS + RVC training half | G.711 @ 20 dB | `lora_norm_rvc_channel` |

Each S3 model is scored in its own condition on the eval pool, CM04 (Tortoise, never
trained on) and the RVC test half, against the same shortcut floors as S2
(`docs/results/w10_norm_rvc.md`). Then S3 **and** the four S2 adapters are scored on
the ASVspoof 2019 LA eval partition. That is the reverse-degradation question:
does training on code-mixed audio cost English performance (W7-T2)?

---

## Before you run

1. **Accelerator: GPU T4 x2. Internet: On.**
2. **+ Add Input**, all three from *Your Work → Datasets*:
   - `saikrishnareddy9/codemix-bundle-normalised` (required: `clean/` and `channel/`)
   - `saikrishnareddy9/asvspoof-2019-la` (required: the English eval partition)
   - `saikrishnareddy9/codemix-w10-results` (optional: the four S2 checkpoints, for
     their English column; set `SCORE_S2_ENGLISH = False` if not attached)
3. Run once with `SMOKE = True`. Then set `SMOKE = False` and use
   **Save Version → Save & Run All**. The full run takes roughly 4-5 hours.

## 1. Configuration

In [ ]:
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH = "week10-mounika-unseen-attack-scoring"   # switch to "main" once merged

SMOKE = True             # True: tiny subsets, confirms the chain end to end
SCORE_S2_ENGLISH = True  # needs codemix-w10-results attached

RUNS = [
    # (name, config, condition)
    ("s3_native_clean",       "configs/train_s3_native_clean.yaml",       "clean"),
    ("s3_native_channel",     "configs/train_s3_native_channel.yaml",     "channel"),
    ("s3_native_rvc_clean",   "configs/train_s3_native_rvc_clean.yaml",   "clean"),
    ("s3_native_rvc_channel", "configs/train_s3_native_rvc_channel.yaml", "channel"),
]

EVALS = {
    "clean":   {"eval_pool": "data/manifests/codemix_eval.csv",
                "cm04":      "data/manifests/score_cm04_norm.csv",
                "rvc_test":  "data/manifests/rvc_holdout_test.csv"},
    "channel": {"eval_pool": "data/manifests/codemix_eval_channel20.csv",
                "cm04":      "data/manifests/score_cm04_norm_channel20.csv",
                "rvc_test":  "data/manifests/rvc_holdout_test_channel20.csv"},
}

S2_ADAPTERS = ["lora_norm_clean", "lora_norm_channel", "lora_norm_rvc_clean", "lora_norm_rvc_channel"]

SCORE_ARGS = "--device cuda --batch-size 32 --num-workers 4 --max-seconds 4.0"

## 2. Clone the repository

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "codemix-deepfake-detection"
OUT = WORK / "s3_results"


def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=REPO if REPO.exists() else None)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode and r.stderr.strip():
        print(r.stderr[-2500:])
    if check and r.returncode:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r


if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", f"https://{REPO_HOST}", str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
OUT.mkdir(exist_ok=True)
sh("git log --oneline -1")
sh(f"{sys.executable} -m pip install -q 'transformers>=4.57,<5' soundfile==0.12.1 librosa==0.11.0", check=False)

## 3. Locate the inputs

Every match is validated rather than trusted by order: `/kaggle/input` is shared by
every attached dataset (P-024).

In [ ]:
import pandas as pd
from src.data.scoring_manifests import missing_files

clean = [p for p in glob.glob("/kaggle/input/**/clean/clips", recursive=True) if os.listdir(p)]
channel = [p for p in glob.glob("/kaggle/input/**/channel/clips", recursive=True) if os.listdir(p)]
if not clean or not channel:
    raise SystemExit("attach saikrishnareddy9/codemix-bundle-normalised (needs clean/ and channel/)")
ROOT = {"clean": str(Path(clean[0]).parent), "channel": str(Path(channel[0]).parent)}
print(ROOT)
for cond, manifests in EVALS.items():
    for name, m in manifests.items():
        gone = missing_files(pd.read_csv(m), ROOT[cond])
        print(f"{cond:8s} {name:10s} missing {len(gone)}")
        assert not gone, f"{m}: {gone[:3]}"

protocols = [p for p in glob.glob("/kaggle/input/**/ASVspoof2019_LA_cm_protocols", recursive=True)
             if (Path(p).parent / "ASVspoof2019_LA_eval" / "flac").is_dir()]
if not protocols:
    raise SystemExit("attach saikrishnareddy9/asvspoof-2019-la")
LA_ROOT = Path(protocols[0]).parent
sh(f"{sys.executable} -m src.data.asvspoof --la-root '{LA_ROOT}' --splits eval --out-dir {WORK}/english")
ENGLISH = WORK / "english" / "asvspoof_eval.csv"
english = pd.read_csv(ENGLISH)
assert len(english) == 71237 and english.speaker.nunique() == 67, "not the published eval protocol"
print("English eval: 71,237 clips / 67 speakers, as published")

S2_CKPT = {}
if SCORE_S2_ENGLISH:
    for name in S2_ADAPTERS:
        hits = [p for p in glob.glob(f"/kaggle/input/**/{name}_best.pt", recursive=True)
                if os.path.getsize(p) > 10_000_000]
        if hits:
            S2_CKPT[name] = hits[0]
    print("S2 checkpoints found:", sorted(S2_CKPT))
    if len(S2_CKPT) < len(S2_ADAPTERS):
        print("!! attach saikrishnareddy9/codemix-w10-results for the full S2 English column")

## 4. Dataset-rule gate: runs before any training

`config_guard` opens each training config's real manifests. It fails on Tortoise,
evaluation-only corpora, eval-pool speakers, RVC test-half speakers, or an S3 run
that starts from the English checkpoint.

In [ ]:
sh(f"{sys.executable} -m pytest tests/test_splits.py tests/test_config_guard.py tests/test_rvc_holdout.py -q")
sh(f"{sys.executable} -m src.training.config_guard " + " ".join(cfg for _, cfg, _ in RUNS))

## 5. Train the four S3 models

In [ ]:
for name, cfg, cond in RUNS:
    t0 = time.time()
    sh(f"{sys.executable} -m src.training.train --config {cfg} --data-root {ROOT[cond]} --device cuda"
       + (" --smoke" if SMOKE else ""))
    print(f"[{name}] trained in {(time.time() - t0) / 60:.1f} min")

## 6. Score S3 on code-mixed audio, each in its own condition

In [ ]:
results = {}


def score(tag, ckpt, manifest, root=None, limit=None):
    cmd = (f"{sys.executable} -m src.training.evaluate --checkpoint '{ckpt}' --manifest '{manifest}' "
           f"{SCORE_ARGS} --scores-out {OUT}/{tag}_scores.csv --out {OUT}/{tag}.json")
    if root:
        cmd += f" --data-root '{root}'"
    if limit:
        cmd += f" --limit {limit}"
    t0 = time.time()
    sh(cmd)
    pooled = json.loads((OUT / f"{tag}.json").read_text()).get("pooled", {})
    results[tag] = pooled
    print(f"{tag:42s} EER {pooled.get('eer', float('nan')) * 100:6.2f}%   ({(time.time() - t0) / 60:.1f} min)")


for name, _, cond in RUNS:
    ckpt = REPO / "checkpoints" / name / "best.pt"
    for ev, manifest in EVALS[cond].items():
        score(f"{name}__{ev}", ckpt, manifest, ROOT[cond], 200 if SMOKE else None)

## 7. English: S3, plus the four S2 adapters for comparison

In [ ]:
english_limit = 2000 if SMOKE else None
for name, _, _ in RUNS:
    score(f"{name}__english", REPO / "checkpoints" / name / "best.pt", ENGLISH, limit=english_limit)
for name, ckpt in S2_CKPT.items():
    score(f"{name}__english", ckpt, ENGLISH, limit=english_limit)

## 8. Summary, and what to download

In [ ]:
rows = [{"model": k.split("__")[0], "set": k.split("__")[1], "EER %": round(v["eer"] * 100, 2)}
        for k, v in results.items()]
table = pd.DataFrame(rows).pivot(index="model", columns="set", values="EER %")
print(table.to_string())

(OUT / "s3_summary.json").write_text(json.dumps(results, indent=2))
mode = "SMOKE (not quotable)" if SMOKE else "full run"
(OUT / "README.md").write_text(
    f"# S3 native training and English scoring\n\nRun mode: **{mode}**. Branch: `{BRANCH}`.\n\n"
    "EER % (lower is better). Code-mixed sets are scored in each model's own condition.\n\n"
    "```\n" + table.to_string() + "\n```\n\n"
    "Floors: `docs/results/w10_norm_rvc.md`. Per-clip scores and pooled JSONs sit beside this file.\n"
)
for name, _, _ in RUNS:
    shutil.copy(REPO / "checkpoints" / name / "best.pt", OUT / f"{name}_best.pt")
print("\nDownload s3_results/ from the Output tab: JSONs, per-clip scores and the four S3 best.pt files.")